# 📝 Notebook 2 — DeBERTa-v3 Training
## NLP Fraud Intent Detection
**Project:** Multimodal Risk Assessment in FinTech Applications | Team 30

**Runtime:** GPU (T4) | **Est. Time:** 20–35 min

### Steps:
1. Mount Google Drive
2. Install Hugging Face transformers
3. Load DIFrauD + SMS Spam datasets
4. Fine-tune DeBERTa-v3-base
5. Save `deberta_model.pt` to Drive

In [ ]:
# ── STEP 0: Check GPU ────────────────────────────────────────────────────────
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else '❌ No GPU! Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# ── STEP 1: Mount Drive ───────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
import os
SAVE_DIR = '/content/drive/MyDrive/MajorProject_Models'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'✅ Drive mounted. Models → {SAVE_DIR}')

In [ ]:
# ── STEP 2: Install Dependencies ─────────────────────────────────────────────
!pip install -q transformers datasets accelerate scikit-learn seaborn

In [ ]:
# ── STEP 3: Load Datasets ─────────────────────────────────────────────────────
from datasets import load_dataset, concatenate_datasets, Dataset
import pandas as pd

print('Loading DIFrauD dataset from Hugging Face...')
try:
    difraud = load_dataset('redasers/difraud', split='train')
    print(f'✅ DIFrauD loaded: {len(difraud)} samples')
    print(f'   Columns: {difraud.column_names}')
except Exception as e:
    print(f'DIFrauD not available: {e}. Using SMS Spam fallback.')
    difraud = None

print('\nLoading SMS Spam dataset...')
sms = load_dataset('sms_spam', split='train')
print(f'✅ SMS Spam loaded: {len(sms)} samples')

In [ ]:
# ── STEP 4: Preprocess & Combine ─────────────────────────────────────────────
import pandas as pd
from sklearn.model_selection import train_test_split

# Normalize SMS Spam: label 1=spam/fraud, 0=ham/legit
sms_df = pd.DataFrame({'text': sms['sms'], 'label': [1 if l == 1 else 0 for l in sms['label']]})

# Add phishing examples (hardcoded augmentation)
phishing_examples = [
    ('Your account is BLOCKED! Share OTP immediately to unblock.', 1),
    ('URGENT: Unauthorized login detected. Verify your password NOW.', 1),
    ('Congratulations! You won Rs 50,000. Click here to claim.', 1),
    ('Please share your Aadhaar number to receive government subsidy.', 1),
    ('Telugu: Meeru account block ayindi. Ippudu OTP share cheyyandi.', 1),
    ('Your bank transfer of Rs 5000 is complete. Thank you.', 0),
    ('Account balance: Rs 12,500. No action required.', 0),
    ('Your KYC update is pending. Visit branch at your convenience.', 0),
    ('Transaction successful. Ref: TXN123456.', 0),
    ('Your monthly statement is ready. Login to view.', 0),
] * 50

aug_df  = pd.DataFrame(phishing_examples, columns=['text', 'label'])
full_df = pd.concat([sms_df, aug_df], ignore_index=True).sample(frac=1, random_state=42)

if difraud is not None:
    # Try to integrate DIFrauD
    # Assuming columns 'text' and 'label'
    try:
        d_df = difraud.to_pandas()[['text', 'label']].dropna()
        d_df['label'] = (d_df['label'] != 0).astype(int)
        full_df = pd.concat([full_df, d_df.sample(min(2000, len(d_df)))], ignore_index=True)
        print(f'✅ DIFrauD integrated: {len(d_df)} records added')
    except Exception as e:
        print(f'DIFrauD merge skipped: {e}')

train_df, val_df = train_test_split(full_df, test_size=0.15, stratify=full_df['label'], random_state=42)
print(f'\n✅ Dataset ready: Train={len(train_df)} | Val={len(val_df)}')
print(f'   Fraud (1): {full_df.label.sum()} | Legit (0): {(full_df.label==0).sum()}')

In [ ]:
# ── STEP 5: Tokenize ─────────────────────────────────────────────────────────
import torch
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

MODEL_NAME = 'microsoft/deberta-v3-base'
print(f'Loading tokenizer: {MODEL_NAME} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class FraudDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts  = df['text'].tolist()
        self.labels = df['label'].tolist()
        self.enc    = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.enc(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_ds = FraudDataset(train_df, tokenizer)
val_ds   = FraudDataset(val_df, tokenizer)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False, num_workers=2)
print(f'✅ Tokenized | Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

In [ ]:
# ── STEP 6: Load DeBERTa-v3 ──────────────────────────────────────────────────
from transformers import AutoModelForSequenceClassification
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model = model.to(device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ DeBERTa-v3-base loaded | Params: {total/1e6:.1f}M | Trainable: {trainable/1e6:.1f}M')

In [ ]:
# ── STEP 7: Training ─────────────────────────────────────────────────────────
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, roc_auc_score

EPOCHS = 5
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=total_steps//10, num_training_steps=total_steps)

best_f1 = 0.0
history = {'train_loss': [], 'val_f1': [], 'val_auc': []}

for epoch in range(EPOCHS):
    # Train
    model.train()
    total_loss = 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        outputs.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += outputs.loss.item()

    # Validate
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            out = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.softmax(out.logits, dim=1)[:, 1]
            preds = out.logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    f1  = f1_score(all_labels, all_preds, average='binary')
    auc = roc_auc_score(all_labels, all_probs)
    avg_loss = total_loss / len(train_loader)
    history['train_loss'].append(avg_loss)
    history['val_f1'].append(f1)
    history['val_auc'].append(auc)

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), f'{SAVE_DIR}/deberta_model.pt')
        saved = '✅ Saved!'
    else:
        saved = ''

    print(f'Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | F1: {f1:.4f} | AUC: {auc:.4f} {saved}')

print(f'\n🎉 Training done! Best F1: {best_f1:.4f}')
print(f'💾 Model saved: {SAVE_DIR}/deberta_model.pt')

In [ ]:
# ── STEP 8: Plot & Download ───────────────────────────────────────────────────
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history['train_loss'], 'r-o'); axes[0].set_title('Train Loss')
axes[1].plot(history['val_f1'], 'g-o');    axes[1].set_title('Val F1 Score')
axes[2].plot(history['val_auc'], 'b-o');   axes[2].set_title('Val AUC-ROC')
plt.suptitle('DeBERTa-v3 — Fraud Intent Detection Training', fontsize=13)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/deberta_training_plot.png', dpi=150)
plt.show()

from google.colab import files
print('⬇️ Downloading deberta_model.pt...')
files.download(f'{SAVE_DIR}/deberta_model.pt')
print('✅ Done! Place in: d:\\Major Project\\model\\deberta_model.pt')